# Outlining the feedback recieved from quant frame

in nb4 you compute value_scores once outside the loop and use the same value_z for every single month from 2015 to 2026. then in cell 18 you flip the sign because the raw signal lost money. sit with what that flip is actually doing for a sec. your naive signal was "long stocks with good past momentum AND cheap today in 2026" but cheap today usually means the price fell, so value points at historical losers and fights momentum until the combined signal goes negative. then you flip it, and now you're longing stocks with good momentum AND expensive today, which in 2026 basically means stocks whose prices went up a lot over the last 10 years. combine that with the fact that your universe is only S&P 500 survivors as of today, and your strategy is to long stocks that are still in the index in 2026 and whose prices went up from 2015 to 2026. the 0.41 sharpe is almost entirely manufactured by survivorship plus forward-looking value plus the flip that aligned them. the flip is what laundered the look-ahead bias into apparent alpha.

you can prove this by running momentum alone on sp500, no value, no flip. whatever sharpe that gives is you actual honest number

for next steps i'd kill the vlaue factor entirely until you can get point in time data, report momentum only as your baseline, then fix survivorship with historical sp500 membership, then add costs, then reintroduce value done properly

Questions?

1. How am I introducing survivorship bias?
2. How can I incorporate transactions costs?
3. I don't understand what you mean here "also no out of sample split. you continuously backtest on 2015-2026 so any weight choice is fit to the full sample. minimum should be in sample/oos, ideally walk forward"?

Answers

1. you're backtesting only on companies that were good enough to survive 11 years in the index, which is a hidden "only winners" filter that inflates returns. fix is to use historical index membership (norgate has it paid, or you can reconstruct from wikipedia edit history, or just use a broad rule like "all US stocks with >$1B market cap at time t" from a point-in-time source)

2. each rebalance, compute turnover as (weight_df.diff().abs().sum(axis=1)) which gives you how much the portfolio churned that month. then subtract cost_bps * turnover from that month's strat return. start with 10 bps (0.001) per side as a rough equity assumption. even better, rerun the backtest at 0, 5, 10, 20 bps and plot how fast your sharpe dies

3. right now you pick your lookback windows, your decile cutoffs, your factor choices, etc, and then backtest on 2015–2026 and look at the sharpe. the problem is you (the researcher) already know what worked on that period, so every tweak you make is silently fitting to the full sample. the number you get back isn't a prediction, it's a fit. the fix is to develop/tune everything on say 2015–2021, freeze it, then run it once on 2022–2026 and whatever you get there is your honest number. walk-forward is the fancier version: at each point in time you only use data before that point to fit params, then trade the next period, then roll forward simulates actually running live. the whole point is that a sharpe computed on data you touched while building the strategy is not a real sharpe, you need numbers from data you never looked at